In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.impute import KNNImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from ucimlrepo import fetch_ucirepo


In [ ]:
y_col_name = "income"
seed = 42

## 1. データの読み込み

In [ ]:
# fetch dataset 
adult = fetch_ucirepo(id=2) 

In [ ]:
# データ読み込み
df_raw = adult.data.original

In [ ]:
df_raw.head(5)

## 2. 最低限の前処理

In [ ]:
# fnlwgtは除外
df_data = df_raw.drop("fnlwgt", axis=1).copy()

In [ ]:
dict_dtype_map = {
    "num": df_data.select_dtypes(include="number").columns.to_list(),
    "cat": df_data.drop(y_col_name, axis=1).select_dtypes(include=["object","string"]).columns.to_list()
}

In [ ]:
dict_dtype_map

In [ ]:
df_data.head(5)

In [ ]:
(df_data == "?").sum()

### 2.1 ?をnanに変換

In [ ]:
# ?をnanに変換
df_data.replace("?", np.nan, inplace=True)

In [ ]:
(df_data == "?").sum()

In [ ]:
df_data.isnull().sum()

### 2.2 目的変数の表記揺れの修正

In [ ]:
df_data["income"].value_counts()

In [ ]:
# .がついているかどうかで表記ブレがあるため、1,0に変換

df_data["income"] = np.where(df_data["income"].str.startswith(">"), 1, 0)

In [ ]:
df_data.head(3)

### 2.3 カテゴリー型に変換

In [ ]:
df_data.dtypes

In [ ]:
df_data.isnull().sum()

### 2.4 エンコーディング

In [ ]:
# エンコーディングのために、nanはnanのまま保持する
ordinal_encoder = OrdinalEncoder(encoded_missing_value=np.nan)


In [ ]:
for col in dict_dtype_map["cat"]:
    
    df_data[col] = ordinal_encoder.fit_transform(df_data[[col]])

In [ ]:
df_data.head(3)

## 3. ベースモデルの学習

### 3.1 データセットの作成

In [ ]:
X_base = df_data.drop(y_col_name, axis=1)
y_base = df_data[y_col_name]

In [ ]:
# テスト用の20%を切り出す
X_train_val_base, X_test_base, y_train_val_base, y_test_base = train_test_split(
    X_base, y_base, 
    test_size=0.2, 
    random_state=seed,
    stratify=y_base
)

In [ ]:
# 全体の20%にするため、80%のうちの25%を指定
X_train_base, X_val_base, y_train_base, y_val_base = train_test_split(
    X_train_val_base, y_train_val_base, 
    test_size=0.25, 
    random_state=seed,
    stratify=y_train_val_base
)

### 3.2 ベースモデルの学習と評価

In [ ]:
model_base = lgb.LGBMClassifier(
    verbose=-1,
    random_state=seed
)

In [ ]:
model_base.fit(
    X_train_base, y_train_base,
    eval_set=[(X_val_base, y_val_base)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=10), 
        
    ]
)

In [ ]:
y_pred_base = model_base.predict(X_test_base)

In [ ]:
# accuracy_score, precision_score, recall_score

acc_base = accuracy_score(y_test_base, y_pred_base)
precision_base = precision_score(y_test_base, y_pred_base)
recall_base = recall_score(y_test_base, y_pred_base)

print(f"basemodel scores:\nacc:{acc_base}\nprecision:{precision_base}\nrecall:{recall_base}")

## 4. 欠損値補完前処理

In [ ]:
df_impute = df_raw.drop("fnlwgt", axis=1).copy()

In [ ]:
df_impute.replace("?", np.nan, inplace=True)

In [ ]:
df_impute["income"] = np.where(df_impute["income"].str.startswith(">"), 1, 0)

In [ ]:
df_impute.head(5)

In [ ]:
knn_pipeline = Pipeline(steps=[
    # 欠損値(NaN)はそのまま保持して数値化する設定
    ("encoder", OrdinalEncoder(
        encoded_missing_value=np.nan, # 欠損値はそのままnan
        handle_unknown="use_encoded_value",  # 未知のデータに対応
        unknown_value=np.nan                # 未知のデータもnanにする
    )),
    
    #  標準化→KNN imputerが距離によって補完するため、スケールを合わせる
    ("scaler", StandardScaler()),
    
    # nanの補完
    ("imputer", KNNImputer(n_neighbors=10))
])

In [ ]:
X_impute = df_impute.drop(y_col_name, axis=1)
y_impute = df_impute[y_col_name]

In [ ]:
# テスト用の20%を切り出す
X_train_val_impute, X_test_impute, y_train_val_impute, y_test_impute = train_test_split(
    X_impute, y_impute, 
    test_size=0.2, 
    random_state=seed,
    stratify=y_impute
)

# 全体の20%にするため、80%のうちの25%を指定
X_train_impute, X_val_impute, y_train_impute, y_val_impute = train_test_split(
    X_train_val_impute, y_train_val_impute, 
    test_size=0.25, 
    random_state=42,
    stratify=y_train_val_impute
)


In [ ]:
X_train_imputed = knn_pipeline.fit_transform(X_train_impute)

In [ ]:
X_val_imputed = knn_pipeline.transform(X_val_impute)
X_test_imputed = knn_pipeline.transform(X_test_impute)

## 5. 欠損値補完後の特徴量でモデル学習、評価

In [ ]:
model_impute = lgb.LGBMClassifier(
    verbose=-1,
    random_state=seed
)

In [ ]:
model_impute.fit(
    X_train_imputed, y_train_impute,
    eval_set=[(X_val_imputed, y_val_impute)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=10), 
        
    ]
)

In [ ]:
y_pred_impute = model_impute.predict(X_test_imputed)

In [ ]:
acc_impute = accuracy_score(y_test_impute, y_pred_impute)
precision_impute = precision_score(y_test_impute, y_pred_impute)
recall_impute = recall_score(y_test_impute, y_pred_impute)

print(f"imputemodel scores:\nacc:{acc_impute}\nprecision:{precision_impute}\nrecall:{recall_impute}")
